# Notebook 1 — Criação da tabela final

## Objetivo
Este notebook consolida as bases do e-commerce Olist em uma única tabela analítica, pronta para exploração, visualização e modelagem.

## Etapas do processo
1. Instalação e importação das bibliotecas.
2. Carregamento das bases brutas.
3. Tratamentos iniciais e padronização de tipos.
4. Enriquecimento das dimensões e agregações auxiliares.
5. Construção da base analítica final.
6. Criação de métricas derivadas.
7. Filtro da visão final de pedidos entregues.
8. Persistência em arquivo `.csv` e em tabela Spark.

> **Resultado esperado:** uma base única, com informações de pedidos, clientes, produtos, sellers, pagamentos, reviews e localização.


## 1. Instalação das bibliotecas

Nesta etapa são instaladas as bibliotecas utilizadas no processo.  
Como o notebook trabalha principalmente com transformação tabular e datas, o foco está em `pandas` e `numpy`.

> `matplotlib` e `seaborn` aparecem na instalação por serem úteis em análises futuras, embora não sejam utilizados diretamente nesta construção da tabela.


In [0]:
# Instala as bibliotecas necessárias para manipulação e futura análise dos dados
%pip install pandas matplotlib seaborn numpy

## 2. Importação das bibliotecas

Aqui carregamos os pacotes que serão utilizados ao longo do notebook:

- **pandas**: leitura, transformação e junção das tabelas;
- **numpy**: apoio para regras condicionais e cálculos vetorizados.


In [0]:
# Biblioteca principal para transformação tabular
import pandas as pd

# Biblioteca de apoio para operações numéricas e regras condicionais
import numpy as np

## 3. Carregamento das bases

Nesta etapa são lidos os arquivos de origem que compõem o modelo analítico.

### Bases carregadas
- **customers**: informações dos clientes;
- **geolocation**: latitude e longitude por CEP;
- **order_items**: itens dos pedidos;
- **order_payments**: pagamentos por pedido;
- **order_reviews**: avaliações dos pedidos;
- **orders**: cabeçalho e status dos pedidos;
- **products**: atributos dos produtos;
- **sellers**: dados dos vendedores;
- **category_translation**: tradução das categorias de produto.


In [0]:
# Define o diretório base onde os arquivos CSV estão armazenados
path = "/Workspace/Users/kaetanokako23@gmail.com/Kaetano-Rodrigues-Tech_Challege_fase1/pos tech/data/Imputs"

# Lê cada uma das bases brutas que serão combinadas ao longo do notebook
customers = pd.read_csv(path + "olist_customers_dataset.csv")
geolocation = pd.read_csv(path + "olist_geolocation_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
order_payments = pd.read_csv(path + "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(path + "olist_order_reviews_dataset.csv")
orders = pd.read_csv(path + "olist_orders_dataset.csv")
products = pd.read_csv(path + "olist_products_dataset.csv")
sellers = pd.read_csv(path + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(path + "product_category_name_translation.csv")

## 4. Tratamentos iniciais

Antes de realizar os joins, é importante garantir que os tipos estejam corretos, principalmente os campos de data.

### O que é feito aqui
- Conversão das datas da tabela de pedidos;
- Conversão da data limite de envio dos itens;
- Conversão das datas de criação e resposta das reviews.

> O parâmetro `errors="coerce"` transforma valores inválidos em `NaT`, evitando quebra no notebook e facilitando o tratamento posterior.


In [0]:
# Lista com as colunas de data presentes na tabela de pedidos
date_cols_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Converte as colunas de data da tabela de pedidos para o tipo datetime
for col in date_cols_orders:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Converte a data limite de envio dos itens
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

# Converte as datas relacionadas às avaliações dos pedidos
order_reviews["review_creation_date"] = pd.to_datetime(
    order_reviews["review_creation_date"],
    errors="coerce"
)
order_reviews["review_answer_timestamp"] = pd.to_datetime(
    order_reviews["review_answer_timestamp"],
    errors="coerce"
)

## 5. Validação rápida das bases

Aqui fazemos uma conferência simples das dimensões de cada tabela para validar se os arquivos foram carregados corretamente.

Essa checagem ajuda a identificar rapidamente:
- arquivos vazios;
- leitura incorreta;
- possíveis divergências de estrutura.


In [0]:
# Organiza os DataFrames em um dicionário para facilitar a inspeção
dfs = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

# Exibe a quantidade de linhas e colunas de cada base
for name, df in dfs.items():
    print(f"{name}: {df.shape}")

## 6. Tratamentos de estrutura e enriquecimento

A partir daqui começamos a preparar tabelas auxiliares para facilitar a montagem da base final.

### 6.1 Padronização das categorias de produto
A base de produtos contém o nome da categoria em português.  
Neste passo, fazemos a junção com a tabela de tradução para obter uma categoria mais amigável para análise.


In [0]:
# Junta a tabela de produtos com a tabela de tradução das categorias
products = products.merge(
    category_translation,
    how="left",
    on="product_category_name"
)

# Cria uma coluna padronizada de categoria:
# usa a tradução em inglês quando existir; caso contrário, mantém o nome original
products["product_category"] = products["product_category_name_english"].fillna(
    products["product_category_name"]
)

### 6.2 Agregação de pagamentos

Como um mesmo pedido pode ter mais de um registro de pagamento, primeiro consolidamos essas informações no nível de `order_id`.

### Métricas criadas
- **payment_value_total**: valor total pago no pedido;
- **payment_installments_max**: maior número de parcelas identificado no pedido;
- **payment_types_n**: quantidade de tipos de pagamento utilizados;
- **payment_type_main**: tipo de pagamento com maior valor dentro do pedido.


In [0]:
# Consolida os pagamentos no nível do pedido
payments_agg = (
    order_payments
    .groupby("order_id", as_index=False)
    .agg(
        payment_value_total=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_types_n=("payment_type", "nunique")
    )
)

# Identifica o principal tipo de pagamento por pedido
# Critério: o tipo com maior valor pago dentro do pedido
payment_main = (
    order_payments
    .groupby(["order_id", "payment_type"], as_index=False)["payment_value"]
    .sum()
    .sort_values(["order_id", "payment_value"], ascending=[True, False])
    .drop_duplicates("order_id")
    [["order_id", "payment_type"]]
    .rename(columns={"payment_type": "payment_type_main"})
)

# Junta a informação do tipo principal ao consolidado de pagamentos
payments_agg = payments_agg.merge(payment_main, on="order_id", how="left")

### 6.3 Agregação de reviews

Um pedido pode ter uma ou mais avaliações.  
Por isso, consolidamos a visão de reviews no nível do pedido para facilitar análises de satisfação do cliente.


In [0]:
# Consolida as avaliações no nível do pedido
reviews_agg = (
    order_reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_score_mean=("review_score", "mean"),
        review_score_min=("review_score", "min"),
        review_score_max=("review_score", "max"),
        review_count=("review_id", "nunique")
    )
)

### 6.4 Agregação de geolocalização

A base de geolocalização pode ter múltiplos registros para o mesmo prefixo de CEP.  
Para evitar duplicidades nos joins, criamos uma visão agregada com a média de latitude e longitude por prefixo.


In [0]:
# Consolida latitude e longitude por prefixo de CEP
geo_agg = (
    geolocation
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geolocation_lat=("geolocation_lat", "mean"),
        geolocation_lng=("geolocation_lng", "mean")
    )
)

## 7. Construção da base analítica

Agora realizamos a junção das principais tabelas para formar a base central de análise.

### Ordem dos joins
1. Itens do pedido;
2. Pedido;
3. Cliente;
4. Produto;
5. Seller;
6. Pagamentos agregados;
7. Reviews agregadas.

> A tabela `order_items` foi utilizada como ponto de partida porque ela representa o nível mais granular do processo: **item dentro do pedido**.


In [0]:
# Inicia a base analítica a partir do nível de item do pedido
df = order_items.merge(orders, how="left", on="order_id")

# Adiciona os dados do cliente responsável pelo pedido
df = df.merge(customers, how="left", on="customer_id")

# Adiciona atributos do produto comprado
df = df.merge(products, how="left", on="product_id")

# Adiciona informações do seller responsável pelo item
df = df.merge(sellers, how="left", on="seller_id")

# Junta os dados consolidados de pagamento no nível do pedido
df = df.merge(payments_agg, how="left", on="order_id")

# Junta os dados consolidados de avaliação do pedido
df = df.merge(reviews_agg, how="left", on="order_id")

## 8. Enriquecimento geográfico

Nesta etapa adicionamos coordenadas aproximadas para clientes e sellers a partir do prefixo do CEP.

### Resultado
- **customer_lat / customer_lng**: localização média do cliente;
- **seller_lat / seller_lng**: localização média do seller.

Essas colunas são úteis para análises de distância, dispersão geográfica e estudos logísticos.


In [0]:
# Adiciona latitude e longitude do cliente com base no prefixo do CEP
df = df.merge(
    geo_agg,
    how="left",
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix"
).rename(columns={
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng"
}).drop(columns=["geolocation_zip_code_prefix"], errors="ignore")

# Renomeia as colunas agregadas de geolocalização para representar o seller
geo_seller = geo_agg.rename(columns={
    "geolocation_zip_code_prefix": "seller_zip_code_prefix",
    "geolocation_lat": "seller_lat",
    "geolocation_lng": "seller_lng"
})

# Adiciona latitude e longitude do seller
df = df.merge(geo_seller, how="left", on="seller_zip_code_prefix")

## 9. Criação das métricas derivadas

Com a base unificada, calculamos indicadores que serão úteis nas análises descritivas e preditivas.

### Métricas financeiras
- **item_revenue**: receita do item;
- **item_freight**: frete do item;
- **item_total**: valor total do item.

### Métricas de prazo e logística
- **approval_time_hours**: tempo entre compra e aprovação;
- **carrier_time_days**: tempo entre aprovação e envio para transportadora;
- **delivery_time_days**: tempo total entre compra e entrega;
- **estimated_time_days**: prazo estimado entre compra e data prevista;
- **delay_days**: atraso real em dias;
- **is_delayed**: flag binária de atraso.

### Métricas temporais
- **purchase_year**;
- **purchase_month**;
- **purchase_year_month**;
- **purchase_quarter**.


In [0]:
# -------------------------
# Métricas financeiras
# -------------------------

# Receita do item sem frete
df["item_revenue"] = df["price"]

# Valor do frete por item
df["item_freight"] = df["freight_value"]

# Valor total do item considerando produto + frete
df["item_total"] = df["price"] + df["freight_value"]

# -------------------------
# Métricas de tempo
# -------------------------

# Tempo entre a compra e a aprovação do pedido, em horas
df["approval_time_hours"] = (
    (df["order_approved_at"] - df["order_purchase_timestamp"]).dt.total_seconds() / 3600
)

# Tempo entre a aprovação e o envio à transportadora, em dias
df["carrier_time_days"] = (
    (df["order_delivered_carrier_date"] - df["order_approved_at"]).dt.total_seconds() / 86400
)

# Tempo total entre a compra e a entrega ao cliente, em dias
df["delivery_time_days"] = (
    (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / 86400
)

# Prazo estimado entre a compra e a data prevista de entrega, em dias
df["estimated_time_days"] = (
    (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / 86400
)

# Diferença entre entrega real e entrega estimada:
# valores positivos indicam atraso
df["delay_days"] = (
    (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.total_seconds() / 86400
)

# Flag binária para identificar pedidos entregues com atraso
df["is_delayed"] = np.where(df["delay_days"] > 0, 1, 0)

# -------------------------
# Métricas temporais
# -------------------------

# Ano da compra
df["purchase_year"] = df["order_purchase_timestamp"].dt.year

# Mês da compra
df["purchase_month"] = df["order_purchase_timestamp"].dt.month

# Ano e mês em formato YYYY-MM
df["purchase_year_month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)

# Trimestre da compra em formato YYYYQn
df["purchase_quarter"] = df["order_purchase_timestamp"].dt.to_period("Q").astype(str)

## 10. Filtro da visão final

Como o objetivo analítico principal está concentrado em pedidos concluídos, criamos uma visão filtrada apenas com pedidos entregues.

> Esse recorte evita distorções em métricas de entrega, atraso e satisfação, já que pedidos cancelados ou em andamento não possuem o ciclo completo.


In [0]:
# Mantém apenas os pedidos efetivamente entregues
df_delivered = df[df["order_status"] == "delivered"].copy()

## 11. Persistência dos dados

Por fim, salvamos a base resultante em dois formatos:

1. **CSV**: útil para compartilhamento e reuso local;
2. **Tabela Spark**: útil para consumo analítico em ambiente distribuído.


In [0]:
# Salva a base tratada em CSV para consumo posterior
df_delivered.to_csv(
    "/Workspace/Users/kaetanokako23@gmail.com/Kaetano-Rodrigues-Tech_Challege_fase1/pos tech/data/base_tratada.csv",
    index=False
)

In [0]:
# Publica a base final como tabela Spark
spark.createDataFrame(df_delivered) \
    .write \
    .mode("overwrite") \
    .saveAsTable("olist_base_analitica")